In this notebook, a recurrent model was trained to forecast ATP player rankings one month ahead using Jeff Sackmann's public dataset and lightweight samples for quick experimentation.

**Workflow overview**\n1. Configure the data source (sample vs. full download).\n2. Load ranking + match records, engineer ranking-focused features, and scale them with `MinMaxScaler`.\n3. Build supervised sequences for LSTM models.\n4. Train single-layer and stacked LSTMs for both player-specific and global setups.\n5. Evaluate with 2024 data and visualise forecasts.

In [0]:
# --- User configuration --------------------------------------------------
use_sample_data = True  # or False to download the full Jeff Sackmann dataset
sequence_length = 8     # number of historical ranking points per training example
forecast_horizon = 1    # predict one ranking step ahead (~4 weeks)
hidden_size = 64        # LSTM hidden units for each layer
epochs = 15             # keep light for demo / sample data
learning_rate = 1e-3
batch_size = 32
selected_player_limit = 6  # how many well-documented players to keep for quick runs
# -------------------------------------------------------------------------

In [0]:
import os
import subprocess
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

plt.style.use('seaborn-v0_8')
warnings.filterwarnings('ignore')

SEED = 24
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = None
current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'data').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = current

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'tennis_atp_raw'
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

print(f'Project root resolved to: {PROJECT_ROOT}')
print(f'Using data directory: {DATA_DIR}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on: {device}')

In [0]:
def ensure_full_dataset(repo_url: str, destination: Path) -> Path:
    destination = destination.expanduser().resolve()
    if destination.exists():
        print('Full dataset already available at', destination)
        return destination
    destination.mkdir(parents=True, exist_ok=True)
    print('Cloning Jeff Sackmann ATP data...')
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(destination)], check=True)
    return destination


def read_rankings_from_repo(repo_dir: Path) -> pd.DataFrame:
    ranking_frames = []
    candidates = ['atp_rankings_20s.csv', 'atp_rankings_10s.csv', 'atp_rankings_00s.csv']
    for file in candidates:
        path = repo_dir / file
        if not path.exists():
            continue
        df = pd.read_csv(path)
        if 'ranking_date' not in df.columns:
            df.columns = ['ranking_date', 'rank', 'player_id', 'player_name', 'country', 'points']
        df = df.rename(columns={'rank': 'ranking'})
        ranking_frames.append(df)
    if not ranking_frames:
        raise FileNotFoundError('No ranking files were found in the cloned repository.')
    ranking_df = pd.concat(ranking_frames, ignore_index=True)
    return ranking_df


def read_matches_from_repo(repo_dir: Path) -> pd.DataFrame:
    matches_frames = []
    for year in range(2022, 2025):
        path = repo_dir / f'atp_matches_{year}.csv'
        if path.exists():
            matches_frames.append(pd.read_csv(path))
    if not matches_frames:
        return pd.DataFrame()
    return pd.concat(matches_frames, ignore_index=True)


if use_sample_data:
    rankings_path = DATA_DIR / 'sample_rankings.csv'
    matches_path = DATA_DIR / 'sample_matches.csv'
    ranking_df = pd.read_csv(rankings_path)
    matches_df = pd.read_csv(matches_path)
    print('Loaded sample data.')
else:
    repo_dir = ensure_full_dataset('https://github.com/JeffSackmann/tennis_atp.git', RAW_DATA_DIR)
    ranking_df = read_rankings_from_repo(repo_dir)
    matches_df = read_matches_from_repo(repo_dir)
    print('Loaded full Jeff Sackmann dataset.')

print(f'Ranking rows: {len(ranking_df):,}')
print(f'Match rows: {len(matches_df):,}')

In [0]:
ranking_df['ranking_date'] = pd.to_datetime(ranking_df['ranking_date'])
if ranking_df['ranking_date'].dt.year.max() < 2024:
    print('Warning: no 2024 rankings in the current dataset. Consider pulling the latest files.')

if 'player_name' not in ranking_df.columns:
    ranking_df['player_name'] = ranking_df['player_id'].astype(str)

if 'ranking' not in ranking_df.columns and 'rank' in ranking_df.columns:
    ranking_df = ranking_df.rename(columns={'rank': 'ranking'})

if 'points' not in ranking_df.columns:
    ranking_df['points'] = np.nan

ranking_df['points'] = ranking_df['points'].fillna(method='ffill')
ranking_df['points'] = ranking_df['points'].fillna(ranking_df['points'].median())

ranking_df = ranking_df.sort_values(['player_id', 'ranking_date'])
ranking_df['rolling_mean_rank'] = (
    ranking_df.groupby('player_id')['ranking']
    .transform(lambda s: s.rolling(window=4, min_periods=1).mean())
)
ranking_df['ranking_velocity'] = ranking_df.groupby('player_id')['ranking'].diff().fillna(0)

matches_df = matches_df.copy()
if not matches_df.empty:
    matches_df['tourney_date'] = pd.to_datetime(matches_df['tourney_date'])
    matches_df['result_value'] = matches_df['result'].map({'W': 1, 'L': 0}).fillna(0.5)
    matches_df['period'] = matches_df['tourney_date'].dt.to_period('M')
    match_features = (
        matches_df.groupby(['player_id', 'period'])
        .agg(matches_played=('result', 'count'), win_rate=('result_value', 'mean'))
        .reset_index()
    )
else:
    match_features = pd.DataFrame(columns=['player_id', 'period', 'matches_played', 'win_rate'])

ranking_df['period'] = ranking_df['ranking_date'].dt.to_period('M')
ranking_df = ranking_df.merge(match_features, how='left', on=['player_id', 'period'])
ranking_df['matches_played'] = ranking_df['matches_played'].fillna(0)
ranking_df['win_rate'] = ranking_df['win_rate'].fillna(ranking_df['win_rate'].mean() if not ranking_df['win_rate'].isna().all() else 0.5)
ranking_df = ranking_df.drop(columns=['period'])

ranking_df['ranking_raw'] = ranking_df['ranking']
feature_cols = ['ranking', 'points', 'rolling_mean_rank', 'ranking_velocity', 'matches_played', 'win_rate']

ranking_df = ranking_df.dropna(subset=['ranking', 'points'])
player_lengths = ranking_df.groupby('player_id')['ranking_date'].count().sort_values(ascending=False)
selected_players = player_lengths.head(selected_player_limit).index.tolist()
ranking_df = ranking_df[ranking_df['player_id'].isin(selected_players)].copy()

train_mask = ranking_df['ranking_date'].dt.year < 2024
feature_scaler = MinMaxScaler()
feature_scaler.fit(ranking_df.loc[train_mask, feature_cols])
ranking_df[feature_cols] = feature_scaler.transform(ranking_df[feature_cols])

ranking_scaler = MinMaxScaler()
ranking_scaler.fit(ranking_df.loc[train_mask, ['ranking_raw']])
ranking_df['ranking_scaled'] = ranking_scaler.transform(ranking_df[['ranking_raw']])

print('Players retained:', len(selected_players))
print(ranking_df[['player_id', 'player_name']].drop_duplicates().head(10))

In [0]:
def build_sequence_records(df: pd.DataFrame,
                          seq_len: int,
                          horizon: int,
                          feature_columns: list[str]) -> list[dict]:
    records = []
    for player_id, group in df.groupby('player_id'):
        group = group.sort_values('ranking_date')
        values = group[feature_columns].values
        targets = group['ranking_scaled'].values
        for start in range(0, len(group) - seq_len - horizon + 1):
            end = start + seq_len
            target_idx = end + horizon - 1
            record = {
                'player_id': player_id,
                'player_name': group['player_name'].iloc[0],
                'target_date': group['ranking_date'].iloc[target_idx],
                'sequence': values[start:end],
                'target': targets[target_idx]
            }
            records.append(record)
    return records


all_records = build_sequence_records(ranking_df, sequence_length, forecast_horizon, feature_cols)
train_records = [r for r in all_records if r['target_date'].year < 2024]
test_records = [r for r in all_records if r['target_date'].year == 2024]

print(f"Total sequences: {len(all_records)} (train={len(train_records)}, test={len(test_records)})")


class SequenceDataset(Dataset):
    def __init__(self, records: list[dict]):
        self.records = records
        self.features = torch.tensor(np.stack([r['sequence'] for r in records]), dtype=torch.float32)
        self.targets = torch.tensor(np.array([r['target'] for r in records]), dtype=torch.float32).unsqueeze(-1)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


def make_loader(records, shuffle=True):
    if not records:
        return None
    dataset = SequenceDataset(records)
    return DataLoader(dataset, batch_size=min(batch_size, len(dataset)), shuffle=shuffle)


def inverse_scale_ranking(values: np.ndarray) -> np.ndarray:
    return ranking_scaler.inverse_transform(values.reshape(-1, 1)).flatten()


In [0]:
class SingleLayerLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, dropout: float = 0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        output = output[:, -1, :]
        output = self.dropout(output)
        return self.fc(output)


class StackedLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                             num_layers=2, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        output = output[:, -1, :]
        output = self.dropout(output)
        return self.fc(output)


def train_model(model: nn.Module, dataloader: DataLoader, epochs: int) -> None:
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    model.to(device)
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        epoch_loss /= len(dataloader.dataset)
        if (epoch + 1) % max(1, epochs // 3) == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch + 1:02d}/{epochs}: L1 loss={epoch_loss:.4f}")


def predict_model(model: nn.Module, records: list[dict]) -> np.ndarray:
    if not records:
        return np.array([])
    dataset = SequenceDataset(records)
    loader = DataLoader(dataset, batch_size=min(batch_size, len(dataset)), shuffle=False)
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            batch_preds = model(xb).cpu().numpy().flatten()
            preds.append(batch_preds)
    return np.concatenate(preds)


def compute_mae(pred_scaled: np.ndarray, true_scaled: np.ndarray) -> float:
    pred = inverse_scale_ranking(pred_scaled)
    true = inverse_scale_ranking(true_scaled)
    return float(np.mean(np.abs(pred - true)))


In [0]:
model_factories = {
    'Single-layer LSTM': lambda: SingleLayerLSTM(len(feature_cols), hidden_size),
    'Stacked LSTM': lambda: StackedLSTM(len(feature_cols), hidden_size)
}

player_results = []
for player_id in sorted({r['player_id'] for r in train_records}):
    player_train = [r for r in train_records if r['player_id'] == player_id]
    player_test = [r for r in test_records if r['player_id'] == player_id]
    if len(player_train) < 5 or len(player_test) == 0:
        continue
    for model_name, factory in model_factories.items():
        model = factory()
        loader = make_loader(player_train, shuffle=True)
        if loader is None:
            continue
        print(f"
Training {model_name} for player {player_train[0]['player_name']} ({player_id})")
        train_model(model, loader, epochs=max(5, epochs // 2))
        preds = predict_model(model, player_test)
        truths = np.array([r['target'] for r in player_test])
        mae = compute_mae(preds, truths)
        player_results.append({
            'player_id': player_id,
            'player_name': player_train[0]['player_name'],
            'model': model_name,
            'mae_rank_points': mae
        })

player_summary = pd.DataFrame(player_results)
if not player_summary.empty:
    display(player_summary)
    display(player_summary.groupby('model')['mae_rank_points'].mean().reset_index().rename(
        columns={'mae_rank_points': 'avg_player_mae'}))
else:
    print('Not enough player-specific data to train individual models. Try pulling the full dataset.')


print('
--- Global models ---')
global_results = []
global_models = {}
train_loader = make_loader(train_records, shuffle=True)
for model_name, factory in model_factories.items():
    model = factory()
    if train_loader is None:
        continue
    train_model(model, train_loader, epochs)
    preds = predict_model(model, test_records)
    truths = np.array([r['target'] for r in test_records])
    mae = compute_mae(preds, truths)
    global_results.append({'model': model_name, 'mae_rank_points': mae})
    global_models[model_name] = model

global_summary = pd.DataFrame(global_results)
if not global_summary.empty:
    display(global_summary)
else:
    print('Global training skipped because there were no sequences available.')


In [0]:
if test_records and global_models:
    best_model_name = min(global_results, key=lambda x: x['mae_rank_points'])['model']
    best_model = global_models[best_model_name]
    example_player_id = test_records[0]['player_id']
    for candidate in test_records:
        if candidate['player_id'] != example_player_id:
            example_player_id = candidate['player_id']
            break
    example_test = [r for r in test_records if r['player_id'] == example_player_id]
    if example_test:
        preds_scaled = predict_model(best_model, example_test)
        truths_scaled = np.array([r['target'] for r in example_test])
        dates = [r['target_date'] for r in example_test]
        preds = inverse_scale_ranking(preds_scaled)
        truths = inverse_scale_ranking(truths_scaled)
        plt.figure(figsize=(10, 4))
        plt.plot(dates, truths, label='True ranking', marker='o')
        plt.plot(dates, preds, label=f'{best_model_name} prediction', marker='x')
        plt.gca().invert_yaxis()
        plt.title(f'2024 ranking forecast for player {example_test[0]["player_name"]}')
        plt.xlabel('Ranking week')
        plt.ylabel('Ranking position (lower is better)')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No example player had 2024 test windows to plot.')
else:
    print('Prediction plot skipped because no trained global model is available.')


### Next steps\n- Increase the sequence length / epochs after switching to the full dataset.\n- Try player embeddings or additional handcrafted features (surface splits, Elo-style form).\n- Export the trained global model to TorchScript for downstream serving.